## Libraries

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, HistGradientBoostingClassifier

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

from tqdm import tqdm

## Dataset with history

### Load dataset

In [2]:
data_final = pd.read_csv('match_consider.csv')

# Apply EDA results
data_final['HomeMatch1EloDiff'] = data_final['HomeMatch1TeamElo'] - data_final['HomeMatch1OppElo']
data_final['AwayMatch1EloDiff'] = data_final['AwayMatch1TeamElo'] - data_final['AwayMatch1OppElo']
data_final['HomeMatch2EloDiff'] = data_final['HomeMatch2TeamElo'] - data_final['HomeMatch2OppElo']
data_final['AwayMatch2EloDiff'] = data_final['AwayMatch2TeamElo'] - data_final['AwayMatch2OppElo']
data_final['HomeMatch3EloDiff'] = data_final['HomeMatch3TeamElo'] - data_final['HomeMatch3OppElo']
data_final['AwayMatch3EloDiff'] = data_final['AwayMatch3TeamElo'] - data_final['AwayMatch3OppElo']
data_final['HomeMatch4EloDiff'] = data_final['HomeMatch4TeamElo'] - data_final['HomeMatch4OppElo']
data_final['AwayMatch4EloDiff'] = data_final['AwayMatch4TeamElo'] - data_final['AwayMatch4OppElo']
data_final['HomeMatch5EloDiff'] = data_final['HomeMatch5TeamElo'] - data_final['HomeMatch5OppElo']
data_final['AwayMatch5EloDiff'] = data_final['AwayMatch5TeamElo'] - data_final['AwayMatch5OppElo']

data_final.drop(
    ['Prev1HomeFouls', 'Prev3HomeFouls', 'Prev5HomeFouls', 'Prev1AwayFouls', 'Prev3AwayFouls', 'Prev5AwayFouls'],
    axis=1, inplace=True)

print('\nShape of data_final:', data_final.shape)
print('\n----------data_final----------')
print(data_final.head())
print('\n----------Information----------')
print(data_final.info())


Shape of data_final: (137447, 91)

----------data_final----------
   Target       Team  Opponent  MatchTime  HomeAway  HomeMatch1Points  \
0       0    Ipswich   Burnley  2246400.0         0                 0   
1       1  Leicester  Brighton  2246400.0         1                 0   
2       3      Luton   Preston  2246400.0         1                 1   
3       1   Millwall       QPR  2246400.0         1                 1   
4       0    Preston     Luton  2246400.0         0                 0   

   HomeMatch2Points  HomeMatch3Points  HomeMatch4Points  HomeMatch5Points  \
0                 1                 0                 3                 3   
1                 3                 0                 1                 3   
2                 1                 3                 1                 3   
3                 0                 0                 0                 1   
4                 1                 1                 0                 1   

   ...  HomeMatch1EloDiff  Away

### Features and target

In [3]:
X_data = data_final.drop(['Target', 'Team', 'Opponent'], axis=1)
y_data = data_final['Target'].map({3:'Win', 1:'Draw', 0:'Lose'})

le = LabelEncoder()
y_data = pd.Series(le.fit_transform(y_data), index=y_data.index)

print('Length of data:', len(X_data))
print('\n----------X_data----------')
print(X_data.head())
print('\n----------y_data----------')
print(y_data.head())

Length of data: 137447

----------X_data----------
   MatchTime  HomeAway  HomeMatch1Points  HomeMatch2Points  HomeMatch3Points  \
0  2246400.0         0                 0                 1                 0   
1  2246400.0         1                 0                 3                 0   
2  2246400.0         1                 1                 1                 3   
3  2246400.0         1                 1                 0                 0   
4  2246400.0         0                 0                 1                 1   

   HomeMatch4Points  HomeMatch5Points  AwayMatch1Points  AwayMatch2Points  \
0                 3                 3                 3                 0   
1                 1                 3                 1                 0   
2                 1                 3                 3                 0   
3                 0                 1                 3                 3   
4                 0                 1                 1                 3   

   Aw

### Split train and test

In [4]:
def rolling_window_split(data_len, n_splits, batch, train_test_ratio):
    splits = []
    start = 0

    step = int((data_len - batch) / (n_splits - 1))
    print('data length:', data_len)
    print('step:', step)
    print()
    
    train_size = int(batch * train_test_ratio)
    test_size = batch - train_size

    while True:
        train_start = start
        train_end = start + train_size
        test_start = train_end
        test_end = train_end + test_size

        if test_end > data_len:
            break

        train_index = np.arange(train_start, train_end)
        test_index = np.arange(test_start, test_end)

        splits.append((train_index, test_index))

        start += step

    return splits


splits = rolling_window_split(len(X_data), 3, 50000, 0.8)
print('# of splits:', len(splits))
print('Size of train data:', len(splits[0][0]))
print('Size of test data:', len(splits[0][1]))
print('\n----------splits----------')
print(splits)

data length: 137447
step: 43723

# of splits: 3
Size of train data: 40000
Size of test data: 10000

----------splits----------
[(array([    0,     1,     2, ..., 39997, 39998, 39999], shape=(40000,)), array([40000, 40001, 40002, ..., 49997, 49998, 49999], shape=(10000,))), (array([43723, 43724, 43725, ..., 83720, 83721, 83722], shape=(40000,)), array([83723, 83724, 83725, ..., 93720, 93721, 93722], shape=(10000,))), (array([ 87446,  87447,  87448, ..., 127443, 127444, 127445],
      shape=(40000,)), array([127446, 127447, 127448, ..., 137443, 137444, 137445],
      shape=(10000,)))]


## Model

### General model definition

In [5]:
models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=2000))
    ]),
    'KNN': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', KNeighborsClassifier(n_neighbors=15))
    ]),
    'SVM': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', SVC(kernel='rbf', probability=True))
    ]),
    'Random Forest': RandomForestClassifier(
        n_estimators=300, min_samples_leaf=5, random_state=1
    )
}

boosting_models = {
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=400,
        learning_rate=0.05,
        max_depth=3,
        random_state=1
    ),
    'AdaBoost': AdaBoostClassifier(
        n_estimators=400,
        learning_rate=0.05,
        random_state=1
    ),
    'HistGradientBoosting': HistGradientBoostingClassifier(
        max_iter=500,
        learning_rate=0.05,
        max_depth=6,
        random_state=1
    )
}

### Model evaluation

In [6]:
def evaluate_single_split(name, model, X_train, X_test, y_train, y_test):
    if name in ['Logistic Regression', 'KNN', 'SVM']:
        X_train.drop(
            ['HomeMatch1TeamElo', 'HomeMatch2TeamElo', 'HomeMatch3TeamElo', 'HomeMatch4TeamElo', 'HomeMatch5TeamElo', 
             'AwayMatch1OppElo', 'AwayMatch2OppElo', 'AwayMatch3OppElo', 'AwayMatch4OppElo', 'AwayMatch5OppElo', 
             'HomeMatch1OppElo', 'HomeMatch2OppElo', 'HomeMatch3OppElo', 'HomeMatch4OppElo', 'HomeMatch5OppElo', 
             'AwayMatch1TeamElo', 'AwayMatch2TeamElo', 'AwayMatch3TeamElo', 'AwayMatch4TeamElo', 'AwayMatch5TeamElo'],
            axis=1, inplace=True
        )
        X_test.drop(
            ['HomeMatch1TeamElo', 'HomeMatch2TeamElo', 'HomeMatch3TeamElo', 'HomeMatch4TeamElo', 'HomeMatch5TeamElo', 
             'AwayMatch1OppElo', 'AwayMatch2OppElo', 'AwayMatch3OppElo', 'AwayMatch4OppElo', 'AwayMatch5OppElo', 
             'HomeMatch1OppElo', 'HomeMatch2OppElo', 'HomeMatch3OppElo', 'HomeMatch4OppElo', 'HomeMatch5OppElo', 
             'AwayMatch1TeamElo', 'AwayMatch2TeamElo', 'AwayMatch3TeamElo', 'AwayMatch4TeamElo', 'AwayMatch5TeamElo'],
            axis=1, inplace=True
        )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average='macro')

    labels = sorted(y_train.unique())
    label_names = le.inverse_transform(labels)
    confusion = confusion_matrix(y_test, y_pred, labels=labels)
    confusion_df = pd.DataFrame(confusion, index=[f'True_{l}' for l in label_names], columns=[f'Pred_{l}' for l in label_names])

    return accuracy, macro_f1, confusion_df

def evalutate_rolling_window(name, model, X, y, splits):
    accuracy_list = []
    macro_f1_list = []

    for train_index, test_index in splits:
        X_train = X.iloc[train_index]
        X_test = X.iloc[test_index]
        y_train = y.iloc[train_index]
        y_test = y.iloc[test_index]

        model.fit(X_train, y_train)
        y_pred - model.predict(X_test)

        accuracy_list.append(accuracy_score(y_test, y_pred))
        macro_f1_list.append(f1_score(y_test, y_pred, average='macro'))

        return {
            'accuracy_mean': np.mean(accuracy_list), 
            'macro_f1_mean': np.mean(f1_macro_list)
        }

In [11]:
train_index, test_index = splits[-1]

for name, model in models.items():
    X_train = X_data.iloc[train_index]
    X_test = X_data.iloc[test_index]
    y_train = y_data.iloc[train_index]
    y_test = y_data.iloc[test_index]
    print(f'\n🚩{name} begins!')
    
    accuracy, macro_f1, confusion = evaluate_single_split(
        name, model, X_train, X_test, y_train, y_test
    )
    print(f'Shape of train/test: {X_train.shape} / {X_test.shape}')
    
    print(f'🚀{name} evaluation completed!')
    print(f'👀Accuracy: {accuracy:.4f}')
    print(f'✨Macro-F1 score: {macro_f1:.4f}')
    print('😎Confusion matrix:')
    print(confusion)

for name, model in boosting_models.items():
    X_train = X_data.iloc[train_index]
    X_test = X_data.iloc[test_index]
    y_train = y_data.iloc[train_index]
    y_test = y_data.iloc[test_index]
    
    print(f'\n🚩{name} begins!')
    accuracy, macro_f1, confusion = evaluate_single_split(
        name, model, X_train, X_test, y_train, y_test
    )
    print(f'Shape of train/test: {X_train.shape} / {X_test.shape}')
    
    print(f'🚀{name} evaluation completed!')
    print(f'👀Accuracy: {accuracy:.4f}')
    print(f'✨Macro-F1 score: {macro_f1:.4f}')
    print('😎Confusion matrix:')
    print(confusion)


🚩Logistic Regression begins!
Shape of train/test: (40000, 68) / (10000, 68)
🚀Logistic Regression evaluation completed!
👀Accuracy: 0.4746
✨Macro-F1 score: 0.3641
😎Confusion matrix:
           Pred_Draw  Pred_Lose  Pred_Win
True_Draw          0       1413      1217
True_Lose          0       2494      1180
True_Win           0       1444      2252

🚩KNN begins!
Shape of train/test: (40000, 68) / (10000, 68)
🚀KNN evaluation completed!
👀Accuracy: 0.4138
✨Macro-F1 score: 0.3903
😎Confusion matrix:
           Pred_Draw  Pred_Lose  Pred_Win
True_Draw        547       1133       950
True_Lose        786       1875      1013
True_Win         681       1299      1716

🚩SVM begins!
Shape of train/test: (40000, 68) / (10000, 68)
🚀SVM evaluation completed!
👀Accuracy: 0.4738
✨Macro-F1 score: 0.3693
😎Confusion matrix:
           Pred_Draw  Pred_Lose  Pred_Win
True_Draw         24       1431      1175
True_Lose         22       2486      1166
True_Win          34       1434      2228

🚩Random Forest b

### Model evaluation after droping 'Draw'

In [7]:
def evaluate_single_split_drop(name, model, X_train, X_test, y_train, y_test):
    if name in ['Logistic Regression', 'KNN', 'SVM']:
        X_train.drop(
            ['HomeMatch1TeamElo', 'HomeMatch2TeamElo', 'HomeMatch3TeamElo', 'HomeMatch4TeamElo', 'HomeMatch5TeamElo', 
             'AwayMatch1OppElo', 'AwayMatch2OppElo', 'AwayMatch3OppElo', 'AwayMatch4OppElo', 'AwayMatch5OppElo', 
             'HomeMatch1OppElo', 'HomeMatch2OppElo', 'HomeMatch3OppElo', 'HomeMatch4OppElo', 'HomeMatch5OppElo', 
             'AwayMatch1TeamElo', 'AwayMatch2TeamElo', 'AwayMatch3TeamElo', 'AwayMatch4TeamElo', 'AwayMatch5TeamElo'],
            axis=1, inplace=True
        )
        X_test.drop(
            ['HomeMatch1TeamElo', 'HomeMatch2TeamElo', 'HomeMatch3TeamElo', 'HomeMatch4TeamElo', 'HomeMatch5TeamElo', 
             'AwayMatch1OppElo', 'AwayMatch2OppElo', 'AwayMatch3OppElo', 'AwayMatch4OppElo', 'AwayMatch5OppElo', 
             'HomeMatch1OppElo', 'HomeMatch2OppElo', 'HomeMatch3OppElo', 'HomeMatch4OppElo', 'HomeMatch5OppElo', 
             'AwayMatch1TeamElo', 'AwayMatch2TeamElo', 'AwayMatch3TeamElo', 'AwayMatch4TeamElo', 'AwayMatch5TeamElo'],
            axis=1, inplace=True
        )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average='macro')

    labels_drop = np.arange(len(le_drop.classes_))
    label_names_drop = le_drop.classes_
    confusion = confusion_matrix(y_test, y_pred, labels=labels_drop)
    confusion_df = pd.DataFrame(confusion, index=[f'True_{l}' for l in label_names_drop], columns=[f'Pred_{l}' for l in label_names_drop])

    return accuracy, macro_f1, confusion_df

data_final_drop = data_final.copy()
idx = data_final_drop.index[data_final_drop['Target'] == 1]
data_final_drop.drop(index=idx, inplace=True)

X_data_drop = data_final_drop.drop(['Target', 'Team', 'Opponent'], axis=1)
y_data_drop = data_final_drop['Target'].map({3:'Win', 0:'Lose'})

le_drop = LabelEncoder()
y_data_drop = pd.Series(le_drop.fit_transform(y_data_drop), index=y_data_drop.index)

print('\nShape of data_final_drop:', data_final_drop.shape)
print('\n----------data_final_drop----------')
print(data_final_drop.head())


Shape of data_final_drop: (101483, 91)

----------data_final_drop----------
   Target     Team Opponent  MatchTime  HomeAway  HomeMatch1Points  \
0       0  Ipswich  Burnley  2246400.0         0                 0   
2       3    Luton  Preston  2246400.0         1                 1   
4       0  Preston    Luton  2246400.0         0                 0   
6       0    Stoke  Cardiff  2246400.0         1                 0   
7       3   Wolves    Crewe  2246400.0         0                 1   

   HomeMatch2Points  HomeMatch3Points  HomeMatch4Points  HomeMatch5Points  \
0                 1                 0                 3                 3   
2                 1                 3                 1                 3   
4                 1                 1                 0                 1   
6                 0                 3                 3                 1   
7                 0                 3                 3                 3   

   ...  HomeMatch1EloDiff  AwayMatch1El

In [8]:
splits_drop = rolling_window_split(len(X_data_drop), 3, 50000, 0.8)
print('# of splits:', len(splits_drop))
print('Size of train data:', len(splits_drop[0][0]))
print('Size of test data:', len(splits_drop[0][1]))
print('\n----------splits_drop----------')
print(splits_drop)

data length: 101483
step: 25741

# of splits: 3
Size of train data: 40000
Size of test data: 10000

----------splits_drop----------
[(array([    0,     1,     2, ..., 39997, 39998, 39999], shape=(40000,)), array([40000, 40001, 40002, ..., 49997, 49998, 49999], shape=(10000,))), (array([25741, 25742, 25743, ..., 65738, 65739, 65740], shape=(40000,)), array([65741, 65742, 65743, ..., 75738, 75739, 75740], shape=(10000,))), (array([51482, 51483, 51484, ..., 91479, 91480, 91481], shape=(40000,)), array([ 91482,  91483,  91484, ..., 101479, 101480, 101481],
      shape=(10000,)))]


In [10]:
train_index, test_index = splits_drop[-1]

for name, model in models.items():
    X_train = X_data_drop.iloc[train_index]
    X_test = X_data_drop.iloc[test_index]
    y_train = y_data_drop.iloc[train_index]
    y_test = y_data_drop.iloc[test_index]
    print(f'\n🚩{name} begins!')
    
    accuracy, macro_f1, confusion = evaluate_single_split_drop(
        name, model, X_train, X_test, y_train, y_test
    )
    print(f'Shape of train/test: {X_train.shape} / {X_test.shape}')
    
    print(f'🚀{name} evaluation without \'Draw\' completed!')
    print(f'👀Accuracy: {accuracy:.4f}')
    print(f'✨Macro-F1 score: {macro_f1:.4f}')
    print('😎Confusion matrix:')
    print(confusion)

for name, model in boosting_models.items():
    X_train = X_data.iloc[train_index]
    X_test = X_data.iloc[test_index]
    y_train = y_data.iloc[train_index]
    y_test = y_data.iloc[test_index]
    
    print(f'\n🚩{name} begins!')
    accuracy, macro_f1, confusion = evaluate_single_split_drop(
        name, model, X_train, X_test, y_train, y_test
    )
    print(f'Shape of train/test: {X_train.shape} / {X_test.shape}')
    
    print(f'🚀{name} evaluation without \'Draw\' completed!')
    print(f'👀Accuracy: {accuracy:.4f}')
    print(f'✨Macro-F1 score: {macro_f1:.4f}')
    print('😎Confusion matrix:')
    print(confusion)


🚩Logistic Regression begins!
Shape of train/test: (40000, 68) / (10000, 68)
🚀Logistic Regression evaluation without 'Draw' completed!
👀Accuracy: 0.6429
✨Macro-F1 score: 0.6427
😎Confusion matrix:
           Pred_Lose  Pred_Win
True_Lose       3341      1648
True_Win        1923      3088

🚩KNN begins!
Shape of train/test: (40000, 68) / (10000, 68)
🚀KNN evaluation without 'Draw' completed!
👀Accuracy: 0.6059
✨Macro-F1 score: 0.6057
😎Confusion matrix:
           Pred_Lose  Pred_Win
True_Lose       3151      1838
True_Win        2103      2908

🚩SVM begins!
Shape of train/test: (40000, 68) / (10000, 68)
🚀SVM evaluation without 'Draw' completed!
👀Accuracy: 0.6396
✨Macro-F1 score: 0.6390
😎Confusion matrix:
           Pred_Lose  Pred_Win
True_Lose       3395      1594
True_Win        2010      3001

🚩Random Forest begins!
Shape of train/test: (40000, 88) / (10000, 88)
🚀Random Forest evaluation without 'Draw' completed!
👀Accuracy: 0.6409
✨Macro-F1 score: 0.6402
😎Confusion matrix:
           Pr

In [9]:
train_index, test_index = splits_drop[-1]
for name, model in boosting_models.items():
    X_train = X_data.iloc[train_index]
    X_test = X_data.iloc[test_index]
    y_train = y_data.iloc[train_index]
    y_test = y_data.iloc[test_index]
    
    print(f'\n🚩{name} begins!')
    accuracy, macro_f1, confusion = evaluate_single_split_drop(
        name, model, X_train, X_test, y_train, y_test
    )
    print(f'Shape of train/test: {X_train.shape} / {X_test.shape}')
    
    print(f'🚀{name} evaluation without \'Draw\' completed!')
    print(f'👀Accuracy: {accuracy:.4f}')
    print(f'✨Macro-F1 score: {macro_f1:.4f}')
    print('😎Confusion matrix:')
    print(confusion)


🚩Gradient Boosting begins!
Shape of train/test: (40000, 88) / (10000, 88)
🚀Gradient Boosting evaluation without 'Draw' completed!
👀Accuracy: 0.4600
✨Macro-F1 score: 0.3597
😎Confusion matrix:
           Pred_Lose  Pred_Win
True_Lose         26      1506
True_Win          29      2438

🚩AdaBoost begins!
Shape of train/test: (40000, 88) / (10000, 88)
🚀AdaBoost evaluation without 'Draw' completed!
👀Accuracy: 0.4552
✨Macro-F1 score: 0.3502
😎Confusion matrix:
           Pred_Lose  Pred_Win
True_Lose          0      1440
True_Win           0      2333

🚩HistGradientBoosting begins!
Shape of train/test: (40000, 88) / (10000, 88)
🚀HistGradientBoosting evaluation without 'Draw' completed!
👀Accuracy: 0.4545
✨Macro-F1 score: 0.3587
😎Confusion matrix:
           Pred_Lose  Pred_Win
True_Lose         40      1495
True_Win          63      2435


### Model evaluation with margin-threshold

In [12]:
def margin_threshold(probability, draw_class=0, delta=0.08):
    pred = np.argmax(probability, axis=1)
    
    top2 = np.sort(probability, axis=1)[:, -2:]
    margin = top2[:, 1] - top2[:, 0]

    pred[margin < delta] = draw_class
    return pred

def evaluate_single_split_threshold(model, X_train, X_test, y_train, y_test, delta=0.08, le=None):

    if name in ['Logistic Regression', 'KNN', 'SVM']:
        X_train.drop(
            ['HomeMatch1TeamElo', 'HomeMatch2TeamElo', 'HomeMatch3TeamElo', 'HomeMatch4TeamElo', 'HomeMatch5TeamElo', 
             'AwayMatch1OppElo', 'AwayMatch2OppElo', 'AwayMatch3OppElo', 'AwayMatch4OppElo', 'AwayMatch5OppElo', 
             'HomeMatch1OppElo', 'HomeMatch2OppElo', 'HomeMatch3OppElo', 'HomeMatch4OppElo', 'HomeMatch5OppElo', 
             'AwayMatch1TeamElo', 'AwayMatch2TeamElo', 'AwayMatch3TeamElo', 'AwayMatch4TeamElo', 'AwayMatch5TeamElo'],
            axis=1, inplace=True
        )
        X_test.drop(
            ['HomeMatch1TeamElo', 'HomeMatch2TeamElo', 'HomeMatch3TeamElo', 'HomeMatch4TeamElo', 'HomeMatch5TeamElo', 
             'AwayMatch1OppElo', 'AwayMatch2OppElo', 'AwayMatch3OppElo', 'AwayMatch4OppElo', 'AwayMatch5OppElo', 
             'HomeMatch1OppElo', 'HomeMatch2OppElo', 'HomeMatch3OppElo', 'HomeMatch4OppElo', 'HomeMatch5OppElo', 
             'AwayMatch1TeamElo', 'AwayMatch2TeamElo', 'AwayMatch3TeamElo', 'AwayMatch4TeamElo', 'AwayMatch5TeamElo'],
            axis=1, inplace=True
        )
    
    model.fit(X_train, y_train)

    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X_test)
    else:
        raise ValueError("😭This model doesn't support predict_proba().")
    
    proba = model.predict_proba(X_test)

    if le is not None:
        draw_class = int(np.where(le.classes_ == 'Draw')[0][0])
    else:
        draw_class = 0
    
    y_pred = margin_threshold(proba, draw_class=draw_class, delta=delta)

    accuracy = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average='macro')

    labels = sorted(y_train.unique())
    label_names = le.inverse_transform(labels)
    confusion = confusion_matrix(y_test, y_pred, labels=labels)
    confusion_df = pd.DataFrame(confusion, index=[f'True_{l}' for l in label_names], columns=[f'Pred_{l}' for l in label_names])

    return accuracy, macro_f1, confusion_df

def evalutate_rolling_window_threshold(model, X, y, splits, delta=0.08, draw_class=0):
    accuracy_list = []
    macro_f1_list = []

    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X_test)
    else:
        raise ValueError("😭This model doesn't support predict_proba().")

    if le is not None:
        draw_class = int(np.where(le.classes_ == 'Draw')[0][0])
    else:
        draw_class = 0

    for train_index, test_index in splits:
        X_train = X.iloc[train_index]
        X_test = X.iloc[test_index]
        y_train = y.iloc[train_index]
        y_test = y.iloc[test_index]

        model.fit(X_train, y_train)
        
        proba = model.predict_proba(X_test)
        y_pred = margin_threshold(proba, draw_class=draw_class, delta=delta)

        accuracy_list.append(accuracy_score(y_test, y_pred))
        macro_f1_list.append(f1_score(y_test, y_pred, average='macro'))

        return {
            'accuracy_mean': np.mean(accuracy_list), 
            'macro_f1_mean': np.mean(f1_macro_list)
        }

In [13]:
train_index, test_index = splits[-1]
delta_setting = 0.08

for name, model in models.items():
    X_train = X_data.iloc[train_index]
    X_test = X_data.iloc[test_index]
    y_train = y_data.iloc[train_index]
    y_test = y_data.iloc[test_index]
    
    print(f'\n🚩{name} begins!')
    accuracy, macro_f1, confusion = evaluate_single_split_threshold(
        model, X_train, X_test, y_train, y_test, delta=delta_setting, le=le
    )
    print(f'Shape of train/test: {X_train.shape} / {X_test.shape}')
    
    print(f'🚀{name} evaluation with threshold completed!')
    print(f'👀Accuracy: {accuracy:.4f}')
    print(f'✨Macro-F1 score: {macro_f1:.4f}')
    print('😎Confusion matrix:')
    print(confusion)

for name, model in boosting_models.items():
    X_train = X_data.iloc[train_index]
    X_test = X_data.iloc[test_index]
    y_train = y_data.iloc[train_index]
    y_test = y_data.iloc[test_index]
    
    print(f'\n🚩{name} begins!')
    accuracy, macro_f1, confusion = evaluate_single_split_threshold(
        model, X_train, X_test, y_train, y_test, delta=delta_setting, le=le
    )
    print(f'Shape of train/test: {X_train.shape} / {X_test.shape}')
    
    print(f'🚀{name} with threshold evaluation completed!')
    print(f'👀Accuracy: {accuracy:.4f}')
    print(f'✨Macro-F1 score: {macro_f1:.4f}')
    print('😎Confusion matrix:')
    print(confusion)


🚩Logistic Regression begins!
Shape of train/test: (40000, 68) / (10000, 68)
🚀Logistic Regression evaluation with threshold completed!
👀Accuracy: 0.4550
✨Macro-F1 score: 0.4357
😎Confusion matrix:
           Pred_Draw  Pred_Lose  Pred_Win
True_Draw        708       1050       872
True_Lose        824       2060       790
True_Win         901       1013      1782

🚩KNN begins!
Shape of train/test: (40000, 68) / (10000, 68)
🚀KNN evaluation with threshold completed!
👀Accuracy: 0.3865
✨Macro-F1 score: 0.3879
😎Confusion matrix:
           Pred_Draw  Pred_Lose  Pred_Win
True_Draw       1198        742       690
True_Lose       1670       1321       683
True_Win        1536        814      1346

🚩SVM begins!
Shape of train/test: (40000, 68) / (10000, 68)
🚀SVM evaluation with threshold completed!
👀Accuracy: 0.4516
✨Macro-F1 score: 0.4275
😎Confusion matrix:
           Pred_Draw  Pred_Lose  Pred_Win
True_Draw        613       1035       982
True_Lose        780       1971       923
True_Win      

### Model evaluation with entire dataset

In [ ]:
train_len = int(len(X_data) * 0.8) + 1

X_train = X_data.iloc[:train_len]
X_test = X_data.iloc[train_len:]
y_train = y_data.iloc[:train_len]
y_test = y_data.iloc[train_len:]

print('Size of train data:', len(y_train))
print('Size of test data:', len(y_test))

delta_setting = 0.08

for name, model in models.items():
    print(f'\n🚩{name} begins!')
    accuracy, macro_f1, confusion = evaluate_single_split_threshold(
        model, X_train, X_test, y_train, y_test, delta=delta_setting, le=le
    )

    print(f'🚀{name} evaluation completed!')
    print(f'👀Accuracy: {accuracy:.4f}')
    print(f'✨Macro-F1 score: {macro_f1:.4f}')
    print('😎Confusion matrix:')
    print(confusion)

for name, model in boosting_models.items():
    print(f'\n🚩{name} begins!')
    accuracy, macro_f1, confusion = evaluate_single_split_threshold(
        model, X_train, X_test, y_train, y_test, delta=delta_setting, le=le
    )

    print(f'🚀{name} evaluation completed!')
    print(f'👀Accuracy: {accuracy:.4f}')
    print(f'✨Macro-F1 score: {macro_f1:.4f}')
    print('😎Confusion matrix:')
    print(confusion)

In [ ]:
evaluation_results = {}
for name, model in models.items():
    evaluation_results[name] = evaluate_rolling_window(
        model, X_data, y_data, splits
    )
    print(f'🚀{name} evaluation completed!')

model_evaluation = pd.DataFrame(evaluation_results).T